# Unsupervised Anomaly Detection for Outlier Flood Events

## Objective
Apply Isolation Forest and statistical residual modeling to flag atypical waterlogging events occurring under low rainfall or showing anomalous flood duration.

## Input Datasets
- data/processed/mumbai_flood_feature_matrix.csv

---

### Step 1: Isolation Forest Anomaly Detection

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

base_dir = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
proc_dir = os.path.join(base_dir, "data", "processed")
df = pd.read_csv(os.path.join(proc_dir, "mumbai_flood_feature_matrix.csv"))

features = ["rainfall_24h_mm", "recorded_flood_depth_cm", "flood_duration_hours", "elevation_m", "historical_recurrence_rate"]
X = df[features].fillna(df[features].median())

iso = IsolationForest(contamination=0.08, random_state=42)
df["anomaly_flag"] = iso.fit_predict(X)
df["anomaly_score"] = iso.decision_function(X)

anomalies = df[df["anomaly_flag"] == -1]
print(f"Detected {len(anomalies)} anomalous flood records ({round(len(anomalies)/len(df)*100, 1)}% of total).")
print("Sample Anomalous Records (High Depth despite Low Rain or Vice-Versa):")
print(anomalies[["date", "location_name", "rainfall_24h_mm", "recorded_flood_depth_cm", "flood_duration_hours", "anomaly_score"]].head(5))

anomalies.to_csv(os.path.join(proc_dir, "anomalies_detected.csv"), index=False)


Detected 118 anomalous flood records (8.0% of total).
Sample Anomalous Records (High Depth despite Low Rain or Vice-Versa):
         date  ... anomaly_score
0  2010-06-14  ...     -0.030832
2  2010-06-14  ...     -0.091544
3  2010-06-14  ...     -0.058404
5  2010-06-14  ...     -0.015172
6  2010-06-14  ...     -0.010105

[5 rows x 6 columns]


---
## Analytical Limitations & Assumptions
An anomaly score is an investigative diagnostic flag pointing to unobserved operational factors (e.g. pump failure, high-tide flap gate blockage), not physical proof of siltation.

## Key Conclusions
Isolation Forest flagged anomalous incidents where high waterlogging occurred despite modest rainfall, pinpointing candidates for municipal field inspection.